In [ ]:
"""
PSIms Complete Unified System v3.0 - PRODUCTION READY
Pharma Stakeholder Interaction Monitoring System

Author: Subhoit Talukdar
Version: 3.0 - All Issues Fixed + Advanced Features

MAJOR FIXES:
- Fixed engagement score calculation (decimal to percentage conversion)
- Fixed cluster naming using ACTUAL scores instead of scaled centroids
- Fixed eligibility mode application (now dynamic)
- Fixed cluster count application (now dynamic)
- Added zero-data cluster with configurable positioning
- Added file selection for analysis step
- Removed all hardcoded values

NEW FEATURES:
- Smart cluster naming (combined absolute + percentile ranking)
- CSV file selection for analysis
- Cluster quality metrics (silhouette score, recommendations)
- Advanced visualizations (toggle-able)
- Dynamic engagement requirement toggle
- Relative percentile-based ranking
- Comprehensive cluster profiling
- Enhanced error handling
- Progress bars for conversion and analysis
- Automatic kernel cleanup on errors
"""

# Check if running in Jupyter and handle kernel restart
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None and 'IPKernelApp' in ipython.config:
        IS_JUPYTER = True
        # Register automatic cleanup
        import atexit
        def cleanup_on_exit():
            try:
                import gc
                gc.collect()
            except:
                pass
        atexit.register(cleanup_on_exit)
    else:
        IS_JUPYTER = False
except:
    IS_JUPYTER = False

import pandas as pd
import numpy as np
import os
import json
import tkinter as tk
from tkinter import filedialog, messagebox, ttk, scrolledtext
from pathlib import Path
from datetime import datetime
import openpyxl
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from difflib import SequenceMatcher

# Optional visualization libraries
try:
    import matplotlib
    matplotlib.use('Agg')  # Non-interactive backend
    import matplotlib.pyplot as plt
    import seaborn as sns
    VISUALIZATION_AVAILABLE = True
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("Note: matplotlib/seaborn not installed. Visualizations will be disabled.")

import warnings
warnings.filterwarnings('ignore')


# =====================================================
# CONFIGURATION MANAGER
# =====================================================

class PSImsConfig:
    """Manages system configuration with persistent storage"""
    
    def __init__(self):
        self.config_file = 'psims_config.json.backup'
        self.config = self.load_or_create()
    
    def load_or_create(self):
        """Load config or create with default structure"""
        default_config = {
            'folders': {
                'input_folder': '',
                'csv_output': '',
                'results_output': ''
            },
            'settings': {
                'eligibility_mode': 'relaxed',
                'num_clusters': 6,
                'require_engagement': True,
                'zero_data_position': 'first',
                'naming_method': 'combined',
                'high_threshold': 30,
                'low_threshold': 15,
                'show_quality_metrics': True,
                'generate_visualizations': False
            },
            'last_run': None
        }
        
        if os.path.exists(self.config_file):
            try:
                with open(self.config_file, 'r') as f:
                    loaded_config = json.load(f)
                    
                    # Ensure all required keys exist
                    for key in default_config:
                        if key not in loaded_config:
                            loaded_config[key] = default_config[key]
                        elif isinstance(default_config[key], dict):
                            for subkey in default_config[key]:
                                if subkey not in loaded_config[key]:
                                    loaded_config[key][subkey] = default_config[key][subkey]
                    
                    return loaded_config
            except Exception as e:
                print(f"Warning: Config file corrupted, creating new one: {e}")
                return default_config
        
        return default_config
    
    def save(self):
        """Save configuration to file"""
        try:
            with open(self.config_file, 'w') as f:
                json.dump(self.config, f, indent=4)
        except Exception as e:
            print(f"Warning: Could not save config: {e}")
    
    def update_folder(self, key, path):
        """Update folder path in config"""
        if 'folders' not in self.config:
            self.config['folders'] = {}
        self.config['folders'][key] = path
        self.save()
    
    def get_folder(self, key):
        """Get folder path from config"""
        if 'folders' not in self.config:
            self.config['folders'] = {}
        return self.config['folders'].get(key, '')
    
    def update_setting(self, key, value):
        """Update a setting"""
        if 'settings' not in self.config:
            self.config['settings'] = {}
        self.config['settings'][key] = value
        self.save()
    
    def get_setting(self, key, default=None):
        """Get a setting"""
        if 'settings' not in self.config:
            self.config['settings'] = {}
        return self.config['settings'].get(key, default)


# =====================================================
# SCORING CONFIGURATION
# =====================================================

SCORING_CONFIG = {
    'engagement': {
        'email_open_weight': 0.50,
        'email_click_weight': 0.50,
        'whatsapp_read_weight': 0.50,
        'whatsapp_click_weight': 0.50,
        'call_productive_weight': 0.70,
        'call_duration_weight': 0.30,
        'final_email_weight': 0.33,
        'final_whatsapp_weight': 0.33,
        'final_call_weight': 0.34,
        'max_score': 100
    },
    'academic': {
        'publication_points_per_item': 10,
        'publication_max_score': 50,
        'trial_points_per_item': 20,
        'trial_max_score': 30,
        'association_points_per_item': 10,
        'association_max_score': 20,
        'max_score': 100,
        'baseline_threshold': 10  # Academic=10 treated as valid low score
    },
    'social_media': {
        'follower_log_multiplier': 10,
        'follower_max_score': 50,
        'follower_min_threshold': 100,
        'platform_points_per_item': 10,
        'platform_max_score': 30,
        'healthcare_platform_points_per_item': 10,
        'healthcare_platform_max_score': 20,
        'max_score': 100
    },
    'recognition': {
        'award_points_per_item': 15,
        'award_max_score': 30,
        'press_points_per_item': 10,
        'press_max_score': 25,
        'event_points_per_item': 8,
        'event_max_score': 25,
        'association_points_per_item': 5,
        'association_max_score': 20,
        'max_score': 100
    }
}


# =====================================================
# SMART EXCEL CONVERTER
# =====================================================

class SmartConverter:
    """Intelligently combines and converts Excel files"""
    
    def __init__(self, output_folder, log_callback=None):
        self.output_folder = output_folder
        self.log = log_callback or print
        self.warnings = []
    
    def standardize_name(self, name):
        """Standardize names (lowercase, no spaces)"""
        return name.strip().lower().replace(' ', '_').replace("'", "")
    
    def fuzzy_match(self, str1, str2, threshold=0.8):
        """Fuzzy string matching"""
        return SequenceMatcher(None, str1.lower(), str2.lower()).ratio() >= threshold
    
    def combine_pi_batches(self, pi_files):
        """Combine multiple PI batch files - ALLOWS DUPLICATES IN COUNT-BASED SHEETS"""
        self.log("\n📊 Combining Personal Info Batches...")
        
        all_sheets_per_file = {}
        for file in pi_files:
            try:
                wb = openpyxl.load_workbook(file, read_only=True)
                all_sheets_per_file[file] = wb.sheetnames
                wb.close()
                self.log(f"   {os.path.basename(file)}: {len(all_sheets_per_file[file])} sheets")
            except Exception as e:
                self.log(f"   ❌ Error loading {os.path.basename(file)}: {e}")
                continue
        
        if not all_sheets_per_file:
            self.log("   ❌ No valid PI files to process")
            return
        
        master_sheets = all_sheets_per_file[list(all_sheets_per_file.keys())[0]]
        sheet_mapping = {sheet: [] for sheet in master_sheets}
        
        for file in all_sheets_per_file.keys():
            file_sheets = all_sheets_per_file[file]
            for master_sheet in master_sheets:
                if master_sheet in file_sheets:
                    sheet_mapping[master_sheet].append((file, master_sheet))
                else:
                    for file_sheet in file_sheets:
                        if self.fuzzy_match(master_sheet, file_sheet):
                            warning = f"Fuzzy match: '{master_sheet}' → '{file_sheet}' in {os.path.basename(file)}"
                            self.warnings.append(warning)
                            self.log(f"   ⚠️  {warning}")
                            sheet_mapping[master_sheet].append((file, file_sheet))
                            break
        
        # Count-based sheets ALLOW duplicate UINs
        COUNT_BASED_SHEETS = ['publication', 'trials', 'academic_association', 
                             'awards', 'press', 'events', 'association']
        
        self.log("\n   📝 Combining sheets...")
        for sheet_name, file_sheet_pairs in sheet_mapping.items():
            if not file_sheet_pairs:
                self.log(f"   ⚠️  No data found for sheet: {sheet_name}")
                continue
            
            dfs = []
            for file, actual_sheet_name in file_sheet_pairs:
                try:
                    df = pd.read_excel(file, sheet_name=actual_sheet_name)
                    df.columns = [self.standardize_name(col) for col in df.columns]
                    dfs.append(df)
                    self.log(f"      ✓ {os.path.basename(file)} → {actual_sheet_name}: {len(df)} rows")
                except Exception as e:
                    self.log(f"      ❌ Error reading {actual_sheet_name} from {os.path.basename(file)}: {e}")
            
            if dfs:
                combined = pd.concat(dfs, ignore_index=True)
                standardized_sheet = self.standardize_name(sheet_name)
                
                if 'uin' in combined.columns:
                    is_count_based = any(cb_sheet in standardized_sheet for cb_sheet in COUNT_BASED_SHEETS)
                    
                    if not is_count_based and standardized_sheet in ['pi', 'clinics', 'digital', 'healthcare_platforms']:
                        before = len(combined)
                        combined = combined.drop_duplicates(subset=['uin'], keep='last')
                        after = len(combined)
                        if before != after:
                            self.log(f"         Removed {before - after} duplicate UINs (reference sheet)")
                    else:
                        self.log(f"         Kept all {len(combined)} rows (count-based sheet)")
                
                output_name = standardized_sheet + '.csv'
                output_path = os.path.join(self.output_folder, output_name)
                combined.to_csv(output_path, index=False, encoding='utf-8')
                self.log(f"      ✓ Saved {output_name}: {len(combined)} rows, {len(combined.columns)} cols")
    
    def convert_engagement_files(self, engagement_files):
        """Convert engagement files with standardized names"""
        self.log("\n📅 Converting Engagement Files...")
        
        for file in engagement_files:
            try:
                df = pd.read_excel(file)
                df.columns = [self.standardize_name(col) for col in df.columns]
                
                original_name = os.path.basename(file).replace('.xlsx', '').replace('.xls', '')
                standardized_name = self.standardize_name(original_name) + '.csv'
                
                output_path = os.path.join(self.output_folder, standardized_name)
                df.to_csv(output_path, index=False, encoding='utf-8')
                
                self.log(f"   ✓ {original_name} → {standardized_name}: {len(df)} rows")
            except Exception as e:
                self.log(f"   ❌ Failed to convert {os.path.basename(file)}: {e}")
    
    def get_warnings(self):
        return self.warnings


# =====================================================
# PSIMS ANALYSIS ENGINE - FULLY ENHANCED
# =====================================================

class PSImsEngine:
    """Core analytics engine with all enhancements and fixes"""
    
    def __init__(self, csv_folder, output_folder, log_callback, 
             eligibility_mode, require_engagement,
             naming_method, high_threshold, low_threshold,
             show_quality_metrics, generate_visualizations,
             zero_data_position, selected_csv_files):
        
        self.csv_folder = csv_folder
        self.output_folder = output_folder
        self.log = log_callback or print
        self.data = {}
        
        # Dynamic parameters (NOT hardcoded)
        self.eligibility_mode = eligibility_mode
        self.require_engagement = require_engagement
        self.naming_method = naming_method
        self.high_threshold = high_threshold
        self.low_threshold = low_threshold
        self.show_quality_metrics = show_quality_metrics
        self.enable_visualizations = generate_visualizations  # RENAMED to avoid conflict
        self.zero_data_position = zero_data_position
        self.selected_csv_files = selected_csv_files or []
        
        # Eligibility rules
        self.eligibility_rules = {
            'strict': {'min_buckets': 4},
            'moderate': {'min_buckets': 3},
            'relaxed': {'min_buckets': 2},
            'basic': {'min_buckets': 1},
            'custom': {'min_buckets': 2}  # Can be adjusted
        }
        
        self.log(f"🔧 Engine initialized:")
        self.log(f"   Mode: {self.eligibility_mode}")
        self.log(f"   Require Engagement: {self.require_engagement}")
        self.log(f"   Naming Method: {self.naming_method}")
        self.log(f"   Thresholds: High={self.high_threshold}, Low={self.low_threshold}")
    
    def load_csv(self, filename):
        """Safely load CSV with encoding fallback"""
        filepath = os.path.join(self.csv_folder, filename)
        try:
            if os.path.exists(filepath):
                df = pd.read_csv(filepath, encoding='utf-8', low_memory=False)
                return df
            return pd.DataFrame()
        except:
            try:
                return pd.read_csv(filepath, encoding='latin-1', low_memory=False)
            except:
                return pd.DataFrame()
    
    def should_load_file(self, filename):
        """Check if file should be loaded based on selection"""
        if not self.selected_csv_files or len(self.selected_csv_files) == 0:
            return True  # Load all if none selected
        return filename in self.selected_csv_files
    
    def standardize_uin_column(self, df, is_engagement=False):
        """Standardize UIN column name to lowercase 'uin'"""
        if df.empty:
            return df
        
        for col in df.columns:
            if col.lower() == 'uin':
                if col != 'uin':
                    df = df.rename(columns={col: 'uin'})
                return df
        
        uin_variations = [
            'Client doctor code', 'client doctor code', 'CLIENT DOCTOR CODE',
            'Client_doctor_code', 'client_doctor_code', 'Client_Doctor_Code',
            'Doctor Code', 'doctor code', 'DoctorCode', 'doctorcode',
            'doctor_id', 'DoctorID', 'Doctor_ID',
            'Client Code', 'client code', 'ClientCode'
        ]
        
        for col in df.columns:
            if col in uin_variations:
                df = df.rename(columns={col: 'uin'})
                return df
        
        for col in df.columns:
            col_lower = col.lower().replace('_', '').replace(' ', '')
            if any(term in col_lower for term in ['doctorcode', 'clientcode', 'doctorid']):
                df = df.rename(columns={col: 'uin'})
                return df
        
        return df
    
    def standardize_engagement_columns(self, df):
        """Standardize engagement metric column names"""
        column_mappings = {
            'HCP Email Open Rate': 'hcp_email_open_rate',
            'HCP Email Click Rate': 'hcp_email_click_rate',
            'HCP Whatsapp Read Rate': 'hcp_whatsapp_read_rate',
            'HCP Whatsapp Click Rate': 'hcp_whatsapp_click_rate',
            'HCP Call Productive Rate': 'hcp_call_productive_rate',
            'Average Duration Connected Calls': 'average_duration_connected_calls'
        }
        
        rename_dict = {}
        for col in df.columns:
            for original, standardized in column_mappings.items():
                if col.lower() == original.lower():
                    rename_dict[col] = standardized
                    break
        
        if rename_dict:
            df = df.rename(columns=rename_dict)
        
        return df
    
    def load_all_data(self):
        """Load all CSV files with file selection support"""
        self.log("\n📂 Loading Data Files...")
        
        files = {
            'pi': 'pi.csv',
            'clinics': 'clinics.csv',
            'publications': 'publication.csv',
            'trials': 'trials.csv',
            'academic_associations': 'academic_association.csv',
            'digital': 'digital.csv',
            'healthcare_platforms': 'healthcare_platforms.csv',
            'awards': 'awards.csv',
            'press': 'press.csv',
            'events': 'events.csv',
            'associations': 'association.csv'
        }
        
        for key, filename in files.items():
            if self.should_load_file(filename):
                self.data[key] = self.load_csv(filename)
                if not self.data[key].empty:
                    self.data[key] = self.standardize_uin_column(self.data[key])
                    if 'uin' in self.data[key].columns:
                        self.log(f"   ✓ {key}: {len(self.data[key])} rows")
                    else:
                        self.log(f"   ⚠️ {key}: No UIN column")
                        self.data[key] = pd.DataFrame()
                else:
                    self.data[key] = pd.DataFrame()
            else:
                self.data[key] = pd.DataFrame()
                self.log(f"   ⊘ {key}: Skipped (not selected)")
        
        # Load engagement files
        engagement_files = [f for f in os.listdir(self.csv_folder) 
                          if f.endswith('.csv') and self.should_load_file(f) and
                          any(month in f.lower() for month in ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 
                                                                'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])]
        
        engagement_dfs = []
        for eng_file in engagement_files:
            df = self.load_csv(eng_file)
            if not df.empty:
                self.log(f"   📋 {eng_file}: {len(df)} rows")
                df = self.standardize_uin_column(df, is_engagement=True)
                df = self.standardize_engagement_columns(df)
                
                if 'uin' in df.columns:
                    engagement_dfs.append(df)
                    self.log(f"      ✓ {df['uin'].nunique()} unique UINs found")
        
        if engagement_dfs:
            combined = pd.concat(engagement_dfs, ignore_index=True)
            
            expected_cols = {
                'hcp_email_open_rate': 0,
                'hcp_email_click_rate': 0,
                'hcp_whatsapp_read_rate': 0,
                'hcp_whatsapp_click_rate': 0,
                'hcp_call_productive_rate': 0,
                'average_duration_connected_calls': 0
            }
            
            for col, default in expected_cols.items():
                if col not in combined.columns:
                    combined[col] = default
                else:
                    combined[col] = pd.to_numeric(combined[col], errors='coerce').fillna(default)
            
            agg_dict = {col: 'mean' for col in expected_cols.keys()}
            self.data['engagement'] = combined.groupby('uin', as_index=False).agg(agg_dict)
            self.log(f"   ✓ Engagement (averaged): {len(self.data['engagement'])} UINs")
        else:
            self.data['engagement'] = pd.DataFrame()
    
    def aggregate_by_uin(self):
        """Aggregate all data sources by UIN - COUNTS DUPLICATES IN COUNT-BASED SHEETS"""
        self.log("\n🔗 Aggregating by UIN...")
        
        if self.data['pi'].empty:
            self.log("   ❌ No PI data")
            return pd.DataFrame()
        
        master = self.data['pi'][['uin']].drop_duplicates().copy()
        self.log(f"   Master: {len(master)} UINs")
        
        for metric in ['publication_count', 'trial_count', 'academic_association_count',
                      'award_count', 'platform_count', 'total_followers',
                      'healthcare_platform_count', 'press_count', 'event_count', 
                      'association_count']:
            master[metric] = 0
        
        # Publications - Count ALL rows with publication_type
        if not self.data['publications'].empty and 'uin' in self.data['publications'].columns:
            pubs_df = self.data['publications'].copy()
            if 'publication_type' in pubs_df.columns:
                pubs_df = pubs_df[pubs_df['publication_type'].notna() & (pubs_df['publication_type'] != '')]
            
            if len(pubs_df) > 0:
                pubs = pubs_df.groupby('uin').size().reset_index(name='publication_count')
                master = master.merge(pubs, on='uin', how='left', suffixes=('', '_new'))
                master['publication_count'] = master['publication_count_new'].fillna(0)
                master.drop(['publication_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Publications: {(master['publication_count'] > 0).sum()} doctors, {master['publication_count'].sum():.0f} total")
        
        # Trials
        if not self.data['trials'].empty and 'uin' in self.data['trials'].columns:
            trials_df = self.data['trials'].copy()
            trial_col = None
            for col in ['trial_type', 'trial_id', 'trial_title']:
                if col in trials_df.columns:
                    trial_col = col
                    break
            
            if trial_col:
                trials_df = trials_df[trials_df[trial_col].notna() & (trials_df[trial_col] != '')]
            
            if len(trials_df) > 0:
                trials = trials_df.groupby('uin').size().reset_index(name='trial_count')
                master = master.merge(trials, on='uin', how='left', suffixes=('', '_new'))
                master['trial_count'] = master['trial_count_new'].fillna(0)
                master.drop(['trial_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Trials: {(master['trial_count'] > 0).sum()} doctors")
        
        # Academic Associations
        if not self.data['academic_associations'].empty and 'uin' in self.data['academic_associations'].columns:
            acad_df = self.data['academic_associations'].copy()
            acad_col = None
            for col in ['association_type', 'association_name', 'association_title']:
                if col in acad_df.columns:
                    acad_col = col
                    break
            
            if acad_col:
                acad_df = acad_df[acad_df[acad_col].notna() & (acad_df[acad_col] != '')]
            
            if len(acad_df) > 0:
                acad = acad_df.groupby('uin').size().reset_index(name='academic_association_count')
                master = master.merge(acad, on='uin', how='left', suffixes=('', '_new'))
                master['academic_association_count'] = master['academic_association_count_new'].fillna(0)
                master.drop(['academic_association_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Academic Assoc: {(master['academic_association_count'] > 0).sum()} doctors")
        
        # Awards
        if not self.data['awards'].empty and 'uin' in self.data['awards'].columns:
            awards_df = self.data['awards'].copy()
            if 'award_level' in awards_df.columns:
                awards_df = awards_df[awards_df['award_level'].notna() & (awards_df['award_level'] != '')]
            
            if len(awards_df) > 0:
                awards = awards_df.groupby('uin').size().reset_index(name='award_count')
                master = master.merge(awards, on='uin', how='left', suffixes=('', '_new'))
                master['award_count'] = master['award_count_new'].fillna(0)
                master.drop(['award_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Awards: {(master['award_count'] > 0).sum()} doctors")
        
        # Press
        if not self.data['press'].empty and 'uin' in self.data['press'].columns:
            press_df = self.data['press'].copy()
            if 'press_type' in press_df.columns:
                press_df = press_df[press_df['press_type'].notna() & (press_df['press_type'] != '')]
            
            if len(press_df) > 0:
                press = press_df.groupby('uin').size().reset_index(name='press_count')
                master = master.merge(press, on='uin', how='left', suffixes=('', '_new'))
                master['press_count'] = master['press_count_new'].fillna(0)
                master.drop(['press_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Press: {(master['press_count'] > 0).sum()} doctors")
        
        # Events
        if not self.data['events'].empty and 'uin' in self.data['events'].columns:
            events_df = self.data['events'].copy()
            if 'event_type' in events_df.columns:
                events_df = events_df[events_df['event_type'].notna() & (events_df['event_type'] != '')]
            
            if len(events_df) > 0:
                events = events_df.groupby('uin').size().reset_index(name='event_count')
                master = master.merge(events, on='uin', how='left', suffixes=('', '_new'))
                master['event_count'] = master['event_count_new'].fillna(0)
                master.drop(['event_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Events: {(master['event_count'] > 0).sum()} doctors")
        
        # Associations
        if not self.data['associations'].empty and 'uin' in self.data['associations'].columns:
            assoc_df = self.data['associations'].copy()
            if 'association_type' in assoc_df.columns:
                assoc_df = assoc_df[assoc_df['association_type'].notna() & (assoc_df['association_type'] != '')]
            
            if len(assoc_df) > 0:
                assoc = assoc_df.groupby('uin').size().reset_index(name='association_count')
                master = master.merge(assoc, on='uin', how='left', suffixes=('', '_new'))
                master['association_count'] = master['association_count_new'].fillna(0)
                master.drop(['association_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Associations: {(master['association_count'] > 0).sum()} doctors")
        
        # Digital data
        if not self.data['digital'].empty and 'uin' in self.data['digital'].columns:
            digital_clean = self.data['digital'][(self.data['digital']['uin'].notna()) & 
                                                 (self.data['digital']['uin'] != '')].copy()
            if len(digital_clean) > 0:
                if 'sm_channel' in digital_clean.columns:
                    digital_clean = digital_clean[digital_clean['sm_channel'].notna() & (digital_clean['sm_channel'] != '')]
                    agg_dict = {'platform_count': ('sm_channel', 'nunique')}
                    if 'sm_followers' in digital_clean.columns:
                        digital_clean['sm_followers'] = pd.to_numeric(digital_clean['sm_followers'], errors='coerce').fillna(0)
                        agg_dict['total_followers'] = ('sm_followers', 'sum')
                    digital = digital_clean.groupby('uin').agg(**agg_dict).reset_index()
                else:
                    digital = digital_clean.groupby('uin').size().reset_index(name='platform_count')
                    digital['total_followers'] = 0
                
                master = master.merge(digital, on='uin', how='left', suffixes=('_old', ''))
                master.drop([c for c in master.columns if c.endswith('_old')], axis=1, errors='ignore', inplace=True)
                master['platform_count'] = master.get('platform_count', 0).fillna(0)
                master['total_followers'] = master.get('total_followers', 0).fillna(0)
                self.log(f"   ✓ Social Media: {(master['platform_count'] > 0).sum()} doctors")
        
        # Healthcare platforms
        if not self.data['healthcare_platforms'].empty and 'uin' in self.data['healthcare_platforms'].columns:
            hc = self.data['healthcare_platforms'].groupby('uin').size().reset_index(name='healthcare_platform_count')
            master = master.merge(hc, on='uin', how='left', suffixes=('', '_new'))
            master['healthcare_platform_count'] = master['healthcare_platform_count_new'].fillna(0)
            master.drop(['healthcare_platform_count_new'], axis=1, errors='ignore', inplace=True)
        
        # Merge engagement
        if not self.data['engagement'].empty:
            merged = master.merge(self.data['engagement'], on='uin', how='left')
            for col in ['hcp_email_open_rate', 'hcp_email_click_rate', 'hcp_whatsapp_read_rate',
                       'hcp_whatsapp_click_rate', 'hcp_call_productive_rate', 
                       'average_duration_connected_calls']:
                if col in merged.columns:
                    merged[col] = merged[col].fillna(0)
        else:
            merged = master.copy()
            for col in ['hcp_email_open_rate', 'hcp_email_click_rate', 'hcp_whatsapp_read_rate',
                       'hcp_whatsapp_click_rate', 'hcp_call_productive_rate', 
                       'average_duration_connected_calls']:
                merged[col] = 0
        
        # Merge PI details
        if 'full_name' in self.data['pi'].columns:
            pi_cols = ['uin'] + [c for c in ['full_name', 'mobile', 'whatsapp_phone', 
                                             'email_id_1', 'specialty'] if c in self.data['pi'].columns]
            merged = merged.merge(self.data['pi'][pi_cols].drop_duplicates('uin'), on='uin', how='left')
        
        # Merge clinic data (INCLUDING clinic_city)
        if not self.data['clinics'].empty and 'uin' in self.data['clinics'].columns:
            clinic_cols = ['uin'] + [c for c in ['clinic_address', 'clinic_city', 'clinic_state'] 
                                    if c in self.data['clinics'].columns]
            if len(clinic_cols) > 1:
                merged = merged.merge(self.data['clinics'][clinic_cols].drop_duplicates('uin'), on='uin', how='left')
                self.log(f"   ✓ Merged clinic data (including city)")
        
        merged.fillna(0, inplace=True)
        return merged
    
    def calculate_scores(self, df):
        """Calculate 4 bucket scores - FIXED ENGAGEMENT CALCULATION"""
        self.log("\n🎯 Calculating Scores...")
        
        def calc_engagement(row):
            config = SCORING_CONFIG['engagement']
            
            email_open = row.get('hcp_email_open_rate', 0) or 0
            email_click = row.get('hcp_email_click_rate', 0) or 0
            wa_read = row.get('hcp_whatsapp_read_rate', 0) or 0
            wa_click = row.get('hcp_whatsapp_click_rate', 0) or 0
            call_prod = row.get('hcp_call_productive_rate', 0) or 0
            call_dur = row.get('average_duration_connected_calls', 0) or 0
            
            # CRITICAL FIX: Convert decimals (0-1) to percentages (0-100)
            if 0 < email_open <= 1:
                email_open = email_open * 100
            if 0 < email_click <= 1:
                email_click = email_click * 100
            if 0 < wa_read <= 1:
                wa_read = wa_read * 100
            if 0 < wa_click <= 1:
                wa_click = wa_click * 100
            if 0 < call_prod <= 1:
                call_prod = call_prod * 100
            
            email_score = (email_open * config['email_open_weight'] + 
                          email_click * config['email_click_weight'])
            wa_score = (wa_read * config['whatsapp_read_weight'] + 
                       wa_click * config['whatsapp_click_weight'])
            
            call_dur_norm = min((call_dur / 60) * 100, 100) if call_dur > 0 else 0
            call_score = (call_prod * config['call_productive_weight'] + 
                         call_dur_norm * config['call_duration_weight'])
            
            final = (email_score * config['final_email_weight'] +
                    wa_score * config['final_whatsapp_weight'] +
                    call_score * config['final_call_weight'])
            
            return round(final, 2)
        
        def calc_academic(row):
            config = SCORING_CONFIG['academic']
            pubs = row.get('publication_count', 0) or 0
            trials = row.get('trial_count', 0) or 0
            acad_assoc = row.get('academic_association_count', 0) or 0
            
            pub_score = min(pubs * config['publication_points_per_item'], 
                          config['publication_max_score'])
            trial_score = min(trials * config['trial_points_per_item'], 
                            config['trial_max_score'])
            assoc_score = min(acad_assoc * config['association_points_per_item'], 
                            config['association_max_score'])
            
            total = pub_score + trial_score + assoc_score
            return round(min(total, config['max_score']), 2)
        
        def calc_social(row):
            config = SCORING_CONFIG['social_media']
            platforms = row.get('platform_count', 0) or 0
            followers = row.get('total_followers', 0) or 0
            hc_platforms = row.get('healthcare_platform_count', 0) or 0
            
            if followers >= config['follower_min_threshold']:
                follower_score = min(np.log10(followers) * config['follower_log_multiplier'],
                                   config['follower_max_score'])
            else:
                follower_score = 0
            
            platform_score = min(platforms * config['platform_points_per_item'],
                               config['platform_max_score'])
            hc_score = min(hc_platforms * config['healthcare_platform_points_per_item'],
                         config['healthcare_platform_max_score'])
            
            total = follower_score + platform_score + hc_score
            return round(min(total, config['max_score']), 2)
        
        def calc_recognition(row):
            config = SCORING_CONFIG['recognition']
            awards = row.get('award_count', 0) or 0
            press = row.get('press_count', 0) or 0
            events = row.get('event_count', 0) or 0
            assoc = row.get('association_count', 0) or 0
            
            award_score = min(awards * config['award_points_per_item'],
                           config['award_max_score'])
            press_score = min(press * config['press_points_per_item'],
                           config['press_max_score'])
            event_score = min(events * config['event_points_per_item'],
                           config['event_max_score'])
            assoc_score = min(assoc * config['association_points_per_item'],
                           config['association_max_score'])
            
            total = award_score + press_score + event_score + assoc_score
            return round(min(total, config['max_score']), 2)
        
        df['engagement_score'] = df.apply(calc_engagement, axis=1)
        df['academic_score'] = df.apply(calc_academic, axis=1)
        df['social_media_score'] = df.apply(calc_social, axis=1)
        df['recognition_score'] = df.apply(calc_recognition, axis=1)
        
        # Calculate percentile ranks for relative scoring
        df['engagement_percentile'] = df['engagement_score'].rank(pct=True)
        df['academic_percentile'] = df['academic_score'].rank(pct=True)
        df['social_media_percentile'] = df['social_media_score'].rank(pct=True)
        df['recognition_percentile'] = df['recognition_score'].rank(pct=True)
        
        # Data availability flags
        df['engagement_data_available'] = df['engagement_score'] > 0
        df['academic_data_available'] = (df['publication_count'] + df['trial_count'] + df['academic_association_count']) > 0
        df['social_media_data_available'] = (df['platform_count'] + df['healthcare_platform_count'] + df['total_followers']) > 0
        df['recognition_data_available'] = (df['award_count'] + df['press_count'] + df['event_count'] + df['association_count']) > 0
        
        df['buckets_with_data'] = (df['engagement_data_available'].astype(int) +
                                   df['academic_data_available'].astype(int) +
                                   df['social_media_data_available'].astype(int) +
                                   df['recognition_data_available'].astype(int))
        
        # DYNAMIC eligibility assessment
        rules = self.eligibility_rules.get(self.eligibility_mode, self.eligibility_rules['relaxed'])
        df['eligible_for_clustering'] = (
            (df['buckets_with_data'] >= rules['min_buckets']) &
            (df['engagement_data_available'] if self.require_engagement else True)
        )
        
        def get_reason(row):
            if row['eligible_for_clustering']:
                return ''
            reasons = []
            if self.require_engagement and not row['engagement_data_available']:
                reasons.append('No engagement data')
            if row['buckets_with_data'] < rules['min_buckets']:
                reasons.append(f"Only {int(row['buckets_with_data'])}/{rules['min_buckets']} buckets")
            return '; '.join(reasons)
        
        df['insufficient_data_reason'] = df.apply(get_reason, axis=1)
        
        self.log(f"   ✓ Scored {len(df)} doctors")
        self.log(f"   Avg Engagement: {df['engagement_score'].mean():.2f}")
        self.log(f"   Avg Academic: {df['academic_score'].mean():.2f}")
        self.log(f"   Avg Social Media: {df['social_media_score'].mean():.2f}")
        self.log(f"   Avg Recognition: {df['recognition_score'].mean():.2f}")
        
        return df
    
    def get_smart_cluster_name(self, cluster_data):
        """Generate smart cluster name using ACTUAL scores and percentiles"""
        
        avg_scores = {
            'Engagement': cluster_data['engagement_score'].mean(),
            'Academic': cluster_data['academic_score'].mean(),
            'Social Media': cluster_data['social_media_score'].mean(),
            'Recognition': cluster_data['recognition_score'].mean()
        }
        
        avg_percentiles = {
            'Engagement': cluster_data['engagement_percentile'].mean(),
            'Academic': cluster_data['academic_percentile'].mean(),
            'Social Media': cluster_data['social_media_percentile'].mean(),
            'Recognition': cluster_data['recognition_percentile'].mean()
        }
        
        max_score_bucket = max(avg_scores, key=avg_scores.get)
        max_score = avg_scores[max_score_bucket]
        max_percentile = avg_percentiles[max_score_bucket]
        
        # Check for special cases first
        all_scores = list(avg_scores.values())
        
        # Case 1: All scores very low
        if all(score < self.low_threshold for score in all_scores):
            return "Low Activity Profile"
        
        # Case 2: Only 1 bucket has data
        buckets_with_data = sum(1 for score in all_scores if score > 5)
        if buckets_with_data == 1:
            return f"Single-Bucket: {max_score_bucket}"
        
        # Case 3: Balanced profile (all within 10 points)
        score_range = max(all_scores) - min(all_scores)
        if score_range < 10 and max_score < 25:
            return "Balanced Low Profile"
        elif score_range < 10:
            return "Balanced Profile"
        
        # Case 4: Emerging profile (2-3 buckets, all < 20)
        if buckets_with_data <= 3 and max_score < 20:
            return f"Emerging: {max_score_bucket}"
        
        # Apply naming method
        if self.naming_method == 'absolute':
            # Pure absolute threshold method
            if max_score > self.high_threshold:
                return f"{max_score_bucket}-Focused"
            elif max_score > 20:
                return f"{max_score_bucket}-Leaning"
            else:
                return f"Low {max_score_bucket}"
        
        elif self.naming_method == 'percentile':
            # Pure percentile method
            if max_percentile > 0.90:
                return f"High {max_score_bucket}"
            elif max_percentile > 0.75:
                return f"Above Average {max_score_bucket}"
            elif max_percentile > 0.50:
                return f"Moderate {max_score_bucket}"
            else:
                return f"Low {max_score_bucket}"
        
        else:  # combined (default)
            # Combined: Must meet BOTH absolute AND percentile criteria
            
            # High tier: Score > high_threshold AND top 25%
            if max_score > self.high_threshold and max_percentile > 0.75:
                # Check if dominant (1.5x others)
                other_scores = [s for b, s in avg_scores.items() if b != max_score_bucket]
                if other_scores and max_score > 1.5 * max(other_scores):
                    return f"{max_score_bucket}-Focused"
                else:
                    return f"{max_score_bucket}-Leaning"
            
            # Moderate tier: Score > 20 OR 50-75th percentile
            elif max_score > 20 or max_percentile > 0.50:
                return f"Moderate {max_score_bucket}"
            
            # Low tier
            else:
                return f"Low {max_score_bucket}"
    
    def perform_clustering(self, df, n_clusters=6):
        """FIXED: Cluster with proper naming using ACTUAL scores"""
        self.log(f"\n🔍 Clustering (k={n_clusters}, mode={self.eligibility_mode})...")
        
        # Separate zero-data doctors
        zero_data = df[df['buckets_with_data'] == 0].copy()
        has_data = df[df['buckets_with_data'] > 0].copy()
        
        self.log(f"   Zero-data doctors: {len(zero_data)}")
        self.log(f"   Doctors with data: {len(has_data)}")
        
        cluster_profiles = []
        silhouette_scores = {}
        
        # Handle zero-data cluster
        if len(zero_data) > 0:
            if self.zero_data_position == 'exclude':
                self.log("   ⊘ Zero-data doctors excluded from output")
                # Don't include in result
            else:
                if self.zero_data_position == 'first':
                    zero_data['cluster_id'] = 1
                    zero_data['cluster_name'] = 'Cluster 1: No Data'
                elif self.zero_data_position == 'last':
                    zero_data['cluster_id'] = n_clusters
                    zero_data['cluster_name'] = f'Cluster {n_clusters}: No Data'
                else:  # 'separate' or 'cluster_0'
                    zero_data['cluster_id'] = 0
                    zero_data['cluster_name'] = 'Cluster 0: No Data'
                
                cluster_profiles.append({
                    'cluster_id': zero_data['cluster_id'].iloc[0],
                    'count': len(zero_data),
                    'avg_engagement': 0,
                    'avg_academic': 0,
                    'avg_social_media': 0,
                    'avg_recognition': 0,
                    'cluster_name': zero_data['cluster_name'].iloc[0]
                })
        
        # Cluster doctors with data
        if len(has_data) >= (n_clusters - 1):
            score_cols = ['engagement_score', 'academic_score', 'social_media_score', 'recognition_score']
            for col in score_cols:
                has_data[col] = has_data[col].fillna(0)
            
            features = has_data[score_cols].values
            features = np.nan_to_num(features, nan=0.0)
            
            # CRITICAL: Scale for clustering
            scaler = StandardScaler()
            features_scaled = scaler.fit_transform(features)
            
            # Determine actual number of clusters for K-Means
            if self.zero_data_position == 'exclude' or len(zero_data) == 0:
                kmeans_clusters = n_clusters
                cluster_id_offset = 1
            else:
                kmeans_clusters = n_clusters - 1
                cluster_id_offset = 2 if self.zero_data_position == 'first' else 1
            
            kmeans = KMeans(n_clusters=kmeans_clusters, random_state=42, n_init=10)
            clusters = kmeans.fit_predict(features_scaled)
            has_data['cluster_id'] = clusters + cluster_id_offset
            
            # Calculate silhouette score for quality metrics
            if self.show_quality_metrics and len(has_data) > kmeans_clusters:
                try:
                    silhouette_avg = silhouette_score(features_scaled, clusters)
                    silhouette_scores['overall'] = silhouette_avg
                    self.log(f"   📊 Silhouette Score: {silhouette_avg:.3f}")
                except:
                    pass
            
            # CRITICAL FIX: Name clusters based on ACTUAL scores, not scaled centroids
            for i in range(kmeans_clusters):
                cluster_data = has_data[has_data['cluster_id'] == i + cluster_id_offset]
                
                # Use smart naming with actual scores
                cluster_name = self.get_smart_cluster_name(cluster_data)
                full_cluster_name = f"Cluster {i + cluster_id_offset}: {cluster_name}"
                
                has_data.loc[has_data['cluster_id'] == i + cluster_id_offset, 'cluster_name'] = full_cluster_name
                
                profile = {
                    'cluster_id': i + cluster_id_offset,
                    'count': len(cluster_data),
                    'avg_engagement': cluster_data['engagement_score'].mean(),
                    'avg_academic': cluster_data['academic_score'].mean(),
                    'avg_social_media': cluster_data['social_media_score'].mean(),
                    'avg_recognition': cluster_data['recognition_score'].mean(),
                    'cluster_name': full_cluster_name
                }
                cluster_profiles.append(profile)
                self.log(f"   {full_cluster_name}: {len(cluster_data)} doctors")
        else:
            # Not enough data for clustering
            has_data['cluster_id'] = 1 if self.zero_data_position != 'first' else 2
            has_data['cluster_name'] = f"Cluster {has_data['cluster_id'].iloc[0]}: Mixed Profile"
            cluster_profiles.append({
                'cluster_id': has_data['cluster_id'].iloc[0],
                'count': len(has_data),
                'avg_engagement': has_data['engagement_score'].mean(),
                'avg_academic': has_data['academic_score'].mean(),
                'avg_social_media': has_data['social_media_score'].mean(),
                'avg_recognition': has_data['recognition_score'].mean(),
                'cluster_name': has_data['cluster_name'].iloc[0]
            })
        
        # Combine results
        if self.zero_data_position == 'exclude':
            result = has_data
        else:
            result = pd.concat([zero_data, has_data], ignore_index=True) if len(zero_data) > 0 else has_data
        
        return result, cluster_profiles, silhouette_scores
    
    def create_visualizations(self, df, cluster_profiles):
        """Generate cluster visualizations"""
        if not self.enable_visualizations:
            return None
        
        if not VISUALIZATION_AVAILABLE:
            self.log("\n⚠️  Visualization libraries not installed (matplotlib/seaborn)")
            self.log("   Install with: pip install matplotlib seaborn")
            return None
        
        self.log("\n📈 Generating Visualizations...")
        
        try:
            # Create figure with subplots
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            fig.suptitle('PSIMS Cluster Analysis', fontsize=16, fontweight='bold')
            
            # 1. Cluster Size Distribution
            cluster_counts = df['cluster_name'].value_counts()
            axes[0, 0].barh(range(len(cluster_counts)), cluster_counts.values)
            axes[0, 0].set_yticks(range(len(cluster_counts)))
            axes[0, 0].set_yticklabels([name.split(':')[1].strip() if ':' in name else name 
                                        for name in cluster_counts.index], fontsize=9)
            axes[0, 0].set_xlabel('Number of Physicians')
            axes[0, 0].set_title('Cluster Size Distribution')
            axes[0, 0].grid(axis='x', alpha=0.3)
            
            # 2. Average Scores by Cluster
            profiles_df = pd.DataFrame(cluster_profiles)
            score_cols = ['avg_engagement', 'avg_academic', 'avg_social_media', 'avg_recognition']
            x = np.arange(len(profiles_df))
            width = 0.2
            
            for idx, col in enumerate(score_cols):
                offset = (idx - 1.5) * width
                axes[0, 1].bar(x + offset, profiles_df[col], width, 
                              label=col.replace('avg_', '').replace('_', ' ').title())
            
            axes[0, 1].set_xlabel('Cluster ID')
            axes[0, 1].set_ylabel('Average Score')
            axes[0, 1].set_title('Average Scores by Cluster')
            axes[0, 1].set_xticks(x)
            axes[0, 1].set_xticklabels(profiles_df['cluster_id'])
            axes[0, 1].legend()
            axes[0, 1].grid(axis='y', alpha=0.3)
            
            # 3. Score Distribution (Box Plot)
            has_data = df[df['buckets_with_data'] > 0]
            score_data = [has_data['engagement_score'], has_data['academic_score'],
                         has_data['social_media_score'], has_data['recognition_score']]
            axes[1, 0].boxplot(score_data, labels=['Engagement', 'Academic', 'Social Media', 'Recognition'])
            axes[1, 0].set_ylabel('Score')
            axes[1, 0].set_title('Score Distribution Across All Physicians')
            axes[1, 0].grid(axis='y', alpha=0.3)
            
            # 4. Bucket Data Availability
            bucket_counts = df['buckets_with_data'].value_counts().sort_index()
            axes[1, 1].bar(bucket_counts.index, bucket_counts.values, color='steelblue')
            axes[1, 1].set_xlabel('Number of Buckets with Data')
            axes[1, 1].set_ylabel('Number of Physicians')
            axes[1, 1].set_title('Data Availability Distribution')
            axes[1, 1].grid(axis='y', alpha=0.3)
            
            plt.tight_layout()
            
            # Save figure
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            viz_file = os.path.join(self.output_folder, f'cluster_visualization_{timestamp}.png')
            plt.savefig(viz_file, dpi=300, bbox_inches='tight')
            plt.close()
            
            self.log(f"   ✓ Saved: {viz_file}")
            return viz_file
            
        except Exception as e:
            self.log(f"   ⚠️ Visualization generation failed: {e}")
            return None
    
    def generate_output(self, df, cluster_profiles, silhouette_scores):
        """Save comprehensive results"""
        self.log("\n💾 Generating Output...")
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_file = os.path.join(self.output_folder, f'psims_results_{timestamp}.csv')
        
        # Include all relevant columns
        output_cols = ['uin', 'full_name', 'specialty', 'mobile', 'clinic_city',
                      'engagement_score', 'engagement_percentile', 'engagement_data_available',
                      'academic_score', 'academic_percentile', 'academic_data_available',
                      'social_media_score', 'social_media_percentile', 'social_media_data_available',
                      'recognition_score', 'recognition_percentile', 'recognition_data_available',
                      'buckets_with_data', 'eligible_for_clustering',
                      'cluster_id', 'cluster_name', 'insufficient_data_reason']
        
        output_cols = [c for c in output_cols if c in df.columns]
        output_df = df[output_cols]
        
        output_df.to_csv(output_file, index=False, encoding='utf-8')
        self.log(f"   ✓ Saved: {output_file}")
        
        # Save cluster profiles
        profiles_file = None
        if cluster_profiles:
            profiles_df = pd.DataFrame(cluster_profiles)
            profiles_file = os.path.join(self.output_folder, f'cluster_profiles_{timestamp}.csv')
            profiles_df.to_csv(profiles_file, index=False, encoding='utf-8')
            self.log(f"   ✓ Saved: {profiles_file}")
            
            self.log("\n📊 Cluster Summary:")
            for _, profile in profiles_df.iterrows():
                pct = (profile['count'] / len(df)) * 100
                self.log(f"   • {profile['cluster_name']}: {profile['count']} ({pct:.1f}%)")
                self.log(f"      Eng: {profile['avg_engagement']:.2f} | Acad: {profile['avg_academic']:.2f} | "
                        f"Social: {profile['avg_social_media']:.2f} | Recog: {profile['avg_recognition']:.2f}")
        
        # Generate comprehensive summary
        summary_file = os.path.join(self.output_folder, f'summary_stats_{timestamp}.txt')
        with open(summary_file, 'w', encoding='utf-8') as f:
            f.write("=" * 80 + "\n")
            f.write("PSIMS ANALYSIS SUMMARY REPORT\n")
            f.write("=" * 80 + "\n\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Eligibility Mode: {self.eligibility_mode.upper()}\n")
            f.write(f"Require Engagement: {self.require_engagement}\n")
            f.write(f"Naming Method: {self.naming_method}\n")
            f.write(f"Thresholds: High={self.high_threshold}, Low={self.low_threshold}\n\n")
            
            f.write("-" * 80 + "\n")
            f.write("OVERALL STATISTICS\n")
            f.write("-" * 80 + "\n")
            f.write(f"Total Physicians: {len(df)}\n")
            f.write(f"Eligible for Clustering: {df['eligible_for_clustering'].sum()}\n")
            f.write(f"Insufficient Data: {(~df['eligible_for_clustering']).sum()}\n\n")
            
            f.write("-" * 80 + "\n")
            f.write("DATA AVAILABILITY\n")
            f.write("-" * 80 + "\n")
            for bucket in ['engagement', 'academic', 'social_media', 'recognition']:
                col = f'{bucket}_data_available'
                if col in df.columns:
                    count = df[col].sum()
                    pct = (count / len(df)) * 100
                    f.write(f"{bucket.replace('_', ' ').title()}: {count} ({pct:.1f}%)\n")
            f.write("\n")
            
            f.write("-" * 80 + "\n")
            f.write("SCORE STATISTICS\n")
            f.write("-" * 80 + "\n")
            for bucket in ['engagement', 'academic', 'social_media', 'recognition']:
                score_col = f'{bucket}_score'
                if score_col in df.columns:
                    f.write(f"\n{bucket.replace('_', ' ').title()}:\n")
                    f.write(f"  Mean: {df[score_col].mean():.2f}\n")
                    f.write(f"  Median: {df[score_col].median():.2f}\n")
                    f.write(f"  Std Dev: {df[score_col].std():.2f}\n")
                    f.write(f"  Min: {df[score_col].min():.2f}\n")
                    f.write(f"  Max: {df[score_col].max():.2f}\n")
            
            if cluster_profiles:
                f.write("\n" + "-" * 80 + "\n")
                f.write("CLUSTER DETAILS\n")
                f.write("-" * 80 + "\n")
                for profile in cluster_profiles:
                    pct = (profile['count'] / len(df)) * 100
                    f.write(f"\n{profile['cluster_name']}\n")
                    f.write(f"  Size: {profile['count']} physicians ({pct:.1f}%)\n")
                    f.write(f"  Avg Engagement: {profile['avg_engagement']:.2f}\n")
                    f.write(f"  Avg Academic: {profile['avg_academic']:.2f}\n")
                    f.write(f"  Avg Social Media: {profile['avg_social_media']:.2f}\n")
                    f.write(f"  Avg Recognition: {profile['avg_recognition']:.2f}\n")
            
            if silhouette_scores and self.show_quality_metrics:
                f.write("\n" + "-" * 80 + "\n")
                f.write("CLUSTER QUALITY METRICS\n")
                f.write("-" * 80 + "\n")
                if 'overall' in silhouette_scores:
                    score = silhouette_scores['overall']
                    f.write(f"Silhouette Score: {score:.3f}\n")
                    if score > 0.7:
                        f.write("Quality: Excellent - Strong cluster separation\n")
                    elif score > 0.5:
                        f.write("Quality: Good - Reasonable cluster structure\n")
                    elif score > 0.3:
                        f.write("Quality: Fair - Weak cluster structure\n")
                    else:
                        f.write("Quality: Poor - Consider reducing cluster count\n")
            
            f.write("\n" + "=" * 80 + "\n")
        
        self.log(f"   ✓ Saved: {summary_file}")
        
        # Generate visualizations if enabled
        viz_file = self.create_visualizations(df, cluster_profiles)
        
        return output_file, profiles_file, summary_file, viz_file
    
    def run_analysis(self, n_clusters=6):
        """Execute complete analysis pipeline"""
        self.log("\n" + "="*60)
        self.log("🚀 STARTING PSIMS ANALYSIS")
        self.log("="*60)
        
        self.load_all_data()
        aggregated = self.aggregate_by_uin()
        
        if aggregated.empty:
            self.log("\n❌ No data to analyze")
            return None, None, None, None
        
        scored = self.calculate_scores(aggregated)
        clustered, profiles, silhouette = self.perform_clustering(scored, n_clusters)
        
        output_file, profiles_file, summary_file, viz_file = self.generate_output(clustered, profiles, silhouette)
        
        self.log("\n" + "="*60)
        self.log("✅ ANALYSIS COMPLETE!")
        self.log("="*60)
        
        return output_file, profiles_file, summary_file, viz_file


# =====================================================
# TKINTER GUI APPLICATION - FULLY ENHANCED
# =====================================================

class PSImsGUI:
    """Complete GUI with all advanced features"""
    
    def __init__(self, root):
        self.root = root
        self.root.title("PSIMS v3.0 - Pharma Stakeholder Interaction Monitoring System")
        self.root.geometry("1000x800")
        
        self.config = PSImsConfig()
        
        # File storage
        self.selected_pi_files = []
        self.selected_engagement_files = []
        self.selected_csv_files = []
        self.available_csv_files = []
        
        # Setup UI
        self.create_widgets()
        self.load_saved_paths()
        
    
    def create_widgets(self):
        """Create all UI widgets"""
        
        # Title
        title_frame = tk.Frame(self.root, bg='#2c3e50', height=60)
        title_frame.pack(fill='x')
        title_frame.pack_propagate(False)
        
        tk.Label(title_frame, text="PSIMS v3.0 - Production Ready", 
                font=('Arial', 18, 'bold'), bg='#2c3e50', fg='white').pack(pady=12)
        
        # Scrollable main container
        main_canvas = tk.Canvas(self.root)
        scrollbar = tk.Scrollbar(self.root, orient="vertical", command=main_canvas.yview)
        scrollable_frame = tk.Frame(main_canvas)
        
        scrollable_frame.bind(
            "<Configure>",
            lambda e: main_canvas.configure(scrollregion=main_canvas.bbox("all"))
        )
        
        main_canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        main_canvas.configure(yscrollcommand=scrollbar.set)
        
        main_canvas.pack(side="left", fill="both", expand=True, padx=10, pady=10)
        scrollbar.pack(side="right", fill="y")
        
        # === STEP 1: FOLDER SELECTION ===
        step1_frame = tk.LabelFrame(scrollable_frame, text="Step 1: Select Folders", 
                                    font=('Arial', 11, 'bold'), padx=10, pady=10)
        step1_frame.pack(fill='x', pady=(0, 10))
        
        tk.Label(step1_frame, text="Input Folder:").grid(row=0, column=0, sticky='w', pady=5)
        self.input_folder_var = tk.StringVar()
        tk.Entry(step1_frame, textvariable=self.input_folder_var, width=60).grid(row=0, column=1, padx=5)
        tk.Button(step1_frame, text="Browse", command=self.browse_input).grid(row=0, column=2)
        
        tk.Label(step1_frame, text="CSV Output:").grid(row=1, column=0, sticky='w', pady=5)
        self.csv_folder_var = tk.StringVar()
        tk.Entry(step1_frame, textvariable=self.csv_folder_var, width=60).grid(row=1, column=1, padx=5)
        tk.Button(step1_frame, text="Browse", command=self.browse_csv_output).grid(row=1, column=2)
        
        tk.Label(step1_frame, text="Results Output:").grid(row=2, column=0, sticky='w', pady=5)
        self.results_folder_var = tk.StringVar()
        tk.Entry(step1_frame, textvariable=self.results_folder_var, width=60).grid(row=2, column=1, padx=5)
        tk.Button(step1_frame, text="Browse", command=self.browse_results_output).grid(row=2, column=2)
        
        # === STEP 2: FILE SELECTION & CONVERSION ===
        step2_frame = tk.LabelFrame(scrollable_frame, text="Step 2: Select Files & Convert", 
                                    font=('Arial', 11, 'bold'), padx=10, pady=10)
        step2_frame.pack(fill='x', pady=(0, 10))
        
        pi_frame = tk.Frame(step2_frame)
        pi_frame.pack(fill='x', pady=5)
        tk.Label(pi_frame, text="PI Files:", width=20, anchor='w').pack(side='left', padx=5)
        self.pi_files_label = tk.Label(pi_frame, text="No files selected", fg='gray', anchor='w')
        self.pi_files_label.pack(side='left', padx=5, fill='x', expand=True)
        tk.Button(pi_frame, text="Select PI Files", command=self.select_pi_files,
                 bg='#95a5a6', fg='white', width=15).pack(side='right', padx=5)
        
        eng_frame = tk.Frame(step2_frame)
        eng_frame.pack(fill='x', pady=5)
        tk.Label(eng_frame, text="Engagement Files:", width=20, anchor='w').pack(side='left', padx=5)
        self.eng_files_label = tk.Label(eng_frame, text="No files selected", fg='gray', anchor='w')
        self.eng_files_label.pack(side='left', padx=5, fill='x', expand=True)
        tk.Button(eng_frame, text="Select Engagement Files", command=self.select_engagement_files,
                 bg='#95a5a6', fg='white', width=20).pack(side='right', padx=5)
        
        # Progress bar for conversion
        self.conversion_progress_frame = tk.Frame(step2_frame)
        self.conversion_progress_frame.pack(fill='x', pady=5)
        self.conversion_progress = ttk.Progressbar(self.conversion_progress_frame, 
                                                   mode='indeterminate', length=400)
        self.conversion_status_label = tk.Label(self.conversion_progress_frame, 
                                                text="", fg='blue', font=('Arial', 9))
        
        tk.Button(step2_frame, text="🔄 Convert Files", command=self.convert_files,
                 bg='#3498db', fg='white', font=('Arial', 10, 'bold'),
                 width=25, height=2).pack(pady=10)
        



        # Find this section in the create_widgets method (around line 1450-1480)
        # Replace the Step 3 section with this:

        # === STEP 3: ANALYSIS SETTINGS ===
        step3_frame = tk.LabelFrame(scrollable_frame, text="Step 3: Analysis Configuration", 
                                    font=('Arial', 11, 'bold'), padx=10, pady=10)
        step3_frame.pack(fill='x', pady=(0, 10))

        # CSV File Selection
        csv_select_frame = tk.Frame(step3_frame)
        csv_select_frame.pack(fill='x', pady=5)
        tk.Label(csv_select_frame, text="CSV Files to Analyze:", font=('Arial', 10, 'bold')).pack(anchor='w')
        tk.Button(csv_select_frame, text="Select CSV Files", command=self.select_csv_files,
                bg='#95a5a6', fg='white').pack(anchor='w', pady=5)
        self.csv_files_label = tk.Label(csv_select_frame, text="All files will be used", fg='gray')
        self.csv_files_label.pack(anchor='w')

        # Basic Settings
        basic_settings = tk.Frame(step3_frame)
        basic_settings.pack(fill='x', pady=5)

        tk.Label(basic_settings, text="Eligibility Mode:").grid(row=0, column=0, sticky='w', padx=5, pady=3)
        self.eligibility_var = tk.StringVar(value='relaxed')
        eligibility_combo = ttk.Combobox(basic_settings, textvariable=self.eligibility_var,
                                        values=['strict', 'moderate', 'relaxed', 'basic'],
                                        state='readonly', width=15)
        eligibility_combo.grid(row=0, column=1, padx=5, pady=3)

        self.require_engagement_var = tk.BooleanVar(value=True)
        tk.Checkbutton(basic_settings, text="Require Engagement Data",
                    variable=self.require_engagement_var).grid(row=0, column=2, columnspan=2, sticky='w', padx=20)

        tk.Label(basic_settings, text="Clusters:").grid(row=1, column=0, sticky='w', padx=5, pady=3)
        self.clusters_var = tk.StringVar(value='6')
        tk.Spinbox(basic_settings, from_=3, to=10, textvariable=self.clusters_var, 
                width=10).grid(row=1, column=1, padx=5, pady=3)

        tk.Label(basic_settings, text="Zero-Data Position:").grid(row=1, column=2, sticky='w', padx=5, pady=3)
        self.zero_position_var = tk.StringVar(value='first')
        zero_combo = ttk.Combobox(basic_settings, textvariable=self.zero_position_var,
                                values=['first', 'last', 'separate', 'exclude'],
                                state='readonly', width=12)
        zero_combo.grid(row=1, column=3, padx=5, pady=3)

        # Advanced Settings
        advanced_frame = tk.LabelFrame(step3_frame, text="Advanced Settings", padx=10, pady=5)
        advanced_frame.pack(fill='x', pady=5)

        adv_row1 = tk.Frame(advanced_frame)
        adv_row1.pack(fill='x', pady=3)

        tk.Label(adv_row1, text="Naming Method:").pack(side='left', padx=5)
        self.naming_method_var = tk.StringVar(value='combined')
        naming_combo = ttk.Combobox(adv_row1, textvariable=self.naming_method_var,
                                    values=['absolute', 'percentile', 'combined'],
                                    state='readonly', width=12)
        naming_combo.pack(side='left', padx=5)

        tk.Label(adv_row1, text="High Threshold:").pack(side='left', padx=(20, 5))
        self.high_threshold_var = tk.StringVar(value='30')
        tk.Spinbox(adv_row1, from_=20, to=50, textvariable=self.high_threshold_var,
                width=8).pack(side='left', padx=5)

        tk.Label(adv_row1, text="Low Threshold:").pack(side='left', padx=(20, 5))
        self.low_threshold_var = tk.StringVar(value='15')
        tk.Spinbox(adv_row1, from_=5, to=25, textvariable=self.low_threshold_var,
                width=8).pack(side='left', padx=5)

        # Feature Toggles
        features_frame = tk.Frame(advanced_frame)
        features_frame.pack(fill='x', pady=5)

        self.show_metrics_var = tk.BooleanVar(value=True)
        tk.Checkbutton(features_frame, text="Show Quality Metrics",
                    variable=self.show_metrics_var).pack(side='left', padx=10)

        self.generate_viz_var = tk.BooleanVar(value=False)
        viz_check = tk.Checkbutton(features_frame, text="Generate Visualizations",
                    variable=self.generate_viz_var)
        viz_check.pack(side='left', padx=10)

        if not VISUALIZATION_AVAILABLE:
            viz_check.config(state='disabled')
            tk.Label(features_frame, text="(matplotlib not installed)", 
                    fg='gray', font=('Arial', 8)).pack(side='left')

        # *** ADD THIS NEW SECTION - Progress bar for analysis ***
        self.analysis_progress_frame = tk.Frame(step3_frame)
        self.analysis_progress_frame.pack(fill='x', pady=5)
        self.analysis_progress = ttk.Progressbar(self.analysis_progress_frame, 
                                                mode='indeterminate', length=400)
        self.analysis_status_label = tk.Label(self.analysis_progress_frame, 
                                            text="", fg='blue', font=('Arial', 9))

        # Run Analysis Button
        button_frame = tk.Frame(step3_frame)
        button_frame.pack(pady=10)

        tk.Button(button_frame, text="🔄 Reset to Defaults", command=self.reset_to_defaults,
                bg='#95a5a6', fg='white', font=('Arial', 9),
                width=20, height=1).pack(side='left', padx=5)

        tk.Button(button_frame, text="▶️ Run Analysis", command=self.run_analysis,
                bg='#27ae60', fg='white', font=('Arial', 11, 'bold'),
                width=25, height=2).pack(side='left', padx=5)
        
        
        # CSV File Selection
        csv_select_frame = tk.Frame(step3_frame)
        csv_select_frame.pack(fill='x', pady=5)
        tk.Label(csv_select_frame, text="CSV Files to Analyze:", font=('Arial', 10, 'bold')).pack(anchor='w')
        tk.Button(csv_select_frame, text="Select CSV Files", command=self.select_csv_files,
                 bg='#95a5a6', fg='white').pack(anchor='w', pady=5)
        self.csv_files_label = tk.Label(csv_select_frame, text="All files will be used", fg='gray')
        self.csv_files_label.pack(anchor='w')
        
        # Basic Settings
        basic_settings = tk.Frame(step3_frame)
        basic_settings.pack(fill='x', pady=5)
        
        tk.Label(basic_settings, text="Eligibility Mode:").grid(row=0, column=0, sticky='w', padx=5, pady=3)
        self.eligibility_var = tk.StringVar(value='relaxed')
        eligibility_combo = ttk.Combobox(basic_settings, textvariable=self.eligibility_var,
                                        values=['strict', 'moderate', 'relaxed', 'basic'],
                                        state='readonly', width=15)
        eligibility_combo.grid(row=0, column=1, padx=5, pady=3)
        
        self.require_engagement_var = tk.BooleanVar(value=True)
        tk.Checkbutton(basic_settings, text="Require Engagement Data",
                      variable=self.require_engagement_var).grid(row=0, column=2, columnspan=2, sticky='w', padx=20)
        
        tk.Label(basic_settings, text="Clusters:").grid(row=1, column=0, sticky='w', padx=5, pady=3)
        self.clusters_var = tk.StringVar(value='6')
        tk.Spinbox(basic_settings, from_=3, to=10, textvariable=self.clusters_var, 
                  width=10).grid(row=1, column=1, padx=5, pady=3)
        
        tk.Label(basic_settings, text="Zero-Data Position:").grid(row=1, column=2, sticky='w', padx=5, pady=3)
        self.zero_position_var = tk.StringVar(value='first')
        zero_combo = ttk.Combobox(basic_settings, textvariable=self.zero_position_var,
                                 values=['first', 'last', 'separate', 'exclude'],
                                 state='readonly', width=12)
        zero_combo.grid(row=1, column=3, padx=5, pady=3)
        
        # Advanced Settings
        advanced_frame = tk.LabelFrame(step3_frame, text="Advanced Settings", padx=10, pady=5)
        advanced_frame.pack(fill='x', pady=5)
        
        adv_row1 = tk.Frame(advanced_frame)
        adv_row1.pack(fill='x', pady=3)
        
        tk.Label(adv_row1, text="Naming Method:").pack(side='left', padx=5)
        self.naming_method_var = tk.StringVar(value='combined')
        naming_combo = ttk.Combobox(adv_row1, textvariable=self.naming_method_var,
                                    values=['absolute', 'percentile', 'combined'],
                                    state='readonly', width=12)
        naming_combo.pack(side='left', padx=5)
        
        tk.Label(adv_row1, text="High Threshold:").pack(side='left', padx=(20, 5))
        self.high_threshold_var = tk.StringVar(value='30')
        tk.Spinbox(adv_row1, from_=20, to=50, textvariable=self.high_threshold_var,
                  width=8).pack(side='left', padx=5)
        
        tk.Label(adv_row1, text="Low Threshold:").pack(side='left', padx=(20, 5))
        self.low_threshold_var = tk.StringVar(value='15')
        tk.Spinbox(adv_row1, from_=5, to=25, textvariable=self.low_threshold_var,
                  width=8).pack(side='left', padx=5)
        
        # Feature Toggles
        features_frame = tk.Frame(advanced_frame)
        features_frame.pack(fill='x', pady=5)
        
        self.show_metrics_var = tk.BooleanVar(value=True)
        tk.Checkbutton(features_frame, text="Show Quality Metrics",
                      variable=self.show_metrics_var).pack(side='left', padx=10)
        
        self.generate_viz_var = tk.BooleanVar(value=False)
        viz_check = tk.Checkbutton(features_frame, text="Generate Visualizations",
                      variable=self.generate_viz_var)
        viz_check.pack(side='left', padx=10)
        
        if not VISUALIZATION_AVAILABLE:
            viz_check.config(state='disabled')
            tk.Label(features_frame, text="(matplotlib not installed)", 
                    fg='gray', font=('Arial', 8)).pack(side='left')
        
        # Run Analysis Button
        tk.Button(step3_frame, text="▶️ Run Analysis", command=self.run_analysis,
                 bg='#27ae60', fg='white', font=('Arial', 11, 'bold'),
                 width=25, height=2).pack(pady=10)
        
        # === LOG OUTPUT ===
        log_frame = tk.LabelFrame(scrollable_frame, text="Process Log", 
                                 font=('Arial', 11, 'bold'), padx=10, pady=10)
        log_frame.pack(fill='both', expand=True)
        
        self.log_text = scrolledtext.ScrolledText(log_frame, wrap=tk.WORD, 
                                                  font=('Courier', 9), height=12)
        self.log_text.pack(fill='both', expand=True)
    
    def browse_input(self):
        folder = filedialog.askdirectory()
        if folder:
            self.input_folder_var.set(folder)
            self.config.update_folder('input_folder', folder)
    
    def browse_csv_output(self):
        folder = filedialog.askdirectory()
        if folder:
            self.csv_folder_var.set(folder)
            self.config.update_folder('csv_output', folder)
    
    def browse_results_output(self):
        folder = filedialog.askdirectory()
        if folder:
            self.results_folder_var.set(folder)
            self.config.update_folder('results_output', folder)
    
    def select_pi_files(self):
        initial_dir = self.input_folder_var.get() or os.getcwd()
        files = filedialog.askopenfilenames(
            title="Select PI Batch Files",
            initialdir=initial_dir,
            filetypes=[("Excel files", "*.xlsx *.xls"), ("All files", "*.*")]
        )
        
        if files:
            self.selected_pi_files = list(files)
            file_count = len(self.selected_pi_files)
            file_names = ", ".join([os.path.basename(f) for f in self.selected_pi_files[:2]])
            if file_count > 2:
                file_names += f" ... (+{file_count-2} more)"
            
            self.pi_files_label.config(text=f"{file_count} file(s): {file_names}", fg='green')
            self.log(f"✓ Selected {file_count} PI file(s)")
    
    def select_engagement_files(self):
        initial_dir = self.input_folder_var.get() or os.getcwd()
        files = filedialog.askopenfilenames(
            title="Select Engagement Files",
            initialdir=initial_dir,
            filetypes=[("Excel files", "*.xlsx *.xls"), ("All files", "*.*")]
        )
        
        if files:
            self.selected_engagement_files = list(files)
            file_count = len(self.selected_engagement_files)
            file_names = ", ".join([os.path.basename(f) for f in self.selected_engagement_files[:2]])
            if file_count > 2:
                file_names += f" ... (+{file_count-2} more)"
            
            self.eng_files_label.config(text=f"{file_count} file(s): {file_names}", fg='green')
            self.log(f"✓ Selected {file_count} engagement file(s)")
    
    def select_csv_files(self):
        """Select specific CSV files for analysis"""
        csv_folder = self.csv_folder_var.get()
        
        if not csv_folder or not os.path.exists(csv_folder):
            messagebox.showerror("Error", "Please set CSV output folder first")
            return
        
        # Get available CSV files
        available_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]
        
        if not available_files:
            messagebox.showwarning("Warning", "No CSV files found in folder")
            return
        
        # Create selection dialog
        dialog = tk.Toplevel(self.root)
        dialog.title("Select CSV Files for Analysis")
        dialog.geometry("500x400")
        
        tk.Label(dialog, text="Select CSV files to include in analysis:", 
                font=('Arial', 11, 'bold')).pack(pady=10)
        
        # Listbox with checkboxes
        frame = tk.Frame(dialog)
        frame.pack(fill='both', expand=True, padx=10, pady=10)
        
        scrollbar = tk.Scrollbar(frame)
        scrollbar.pack(side='right', fill='y')
        
        listbox = tk.Listbox(frame, selectmode='multiple', yscrollcommand=scrollbar.set, height=15)
        listbox.pack(side='left', fill='both', expand=True)
        scrollbar.config(command=listbox.yview)
        
        for file in sorted(available_files):
            listbox.insert(tk.END, file)
        
        # Select all by default
        for i in range(len(available_files)):
            listbox.selection_set(i)
        
        button_frame = tk.Frame(dialog)
        button_frame.pack(pady=10)
        
        def select_all():
            listbox.selection_set(0, tk.END)
        
        def deselect_all():
            listbox.selection_clear(0, tk.END)
        
        def confirm():
            selected_indices = listbox.curselection()
            self.selected_csv_files = [available_files[i] for i in selected_indices]
            
            if self.selected_csv_files:
                count = len(self.selected_csv_files)
                self.csv_files_label.config(
                    text=f"{count} CSV file(s) selected for analysis",
                    fg='green'
                )
                self.log(f"✓ Selected {count} CSV files for analysis")
            else:
                self.selected_csv_files = []
                self.csv_files_label.config(text="All files will be used", fg='gray')
            
            dialog.destroy()
        
        tk.Button(button_frame, text="Select All", command=select_all, width=12).pack(side='left', padx=5)
        tk.Button(button_frame, text="Deselect All", command=deselect_all, width=12).pack(side='left', padx=5)
        tk.Button(button_frame, text="Confirm", command=confirm, bg='#27ae60', fg='white', width=12).pack(side='left', padx=5)
    
    def load_saved_paths(self):
        """Load saved folder paths"""
        self.input_folder_var.set(self.config.get_folder('input_folder'))
        self.csv_folder_var.set(self.config.get_folder('csv_output'))
        self.results_folder_var.set(self.config.get_folder('results_output'))
    
    def load_saved_settings(self):
        """Load saved settings"""
        self.eligibility_var.set(self.config.get_setting('eligibility_mode', 'relaxed'))
        self.clusters_var.set(str(self.config.get_setting('num_clusters', 6)))
        self.require_engagement_var.set(self.config.get_setting('require_engagement', True))
        self.zero_position_var.set(self.config.get_setting('zero_data_position', 'first'))
        self.naming_method_var.set(self.config.get_setting('naming_method', 'combined'))
        self.high_threshold_var.set(str(self.config.get_setting('high_threshold', 30)))
        self.low_threshold_var.set(str(self.config.get_setting('low_threshold', 15)))
        self.show_metrics_var.set(self.config.get_setting('show_quality_metrics', True))
        self.generate_viz_var.set(self.config.get_setting('generate_visualizations', False))

    def reset_to_defaults(self):
        """Reset all settings to default values"""
        self.eligibility_var.set('relaxed')
        self.require_engagement_var.set(True)
        self.clusters_var.set('6')
        self.zero_position_var.set('first')
        self.naming_method_var.set('combined')
        self.high_threshold_var.set('30')
        self.low_threshold_var.set('15')
        self.show_metrics_var.set(True)
        self.generate_viz_var.set(False)
        self.log("✓ Settings reset to defaults")
    
    def save_current_settings(self):
        """Save current settings"""
        self.config.update_setting('eligibility_mode', self.eligibility_var.get())
        self.config.update_setting('num_clusters', int(self.clusters_var.get()))
        self.config.update_setting('require_engagement', self.require_engagement_var.get())
        self.config.update_setting('zero_data_position', self.zero_position_var.get())
        self.config.update_setting('naming_method', self.naming_method_var.get())
        self.config.update_setting('high_threshold', int(self.high_threshold_var.get()))
        self.config.update_setting('low_threshold', int(self.low_threshold_var.get()))
        self.config.update_setting('show_quality_metrics', self.show_metrics_var.get())
        self.config.update_setting('generate_visualizations', self.generate_viz_var.get())
    
    def log(self, message):
        """Display message in log"""
        try:
            # Print to console as well (helpful for Jupyter)
            print(message)
            # Also show in GUI
            self.log_text.insert(tk.END, str(message) + '\n')
            self.log_text.see(tk.END)
            self.root.update_idletasks()
        except Exception as e:
            print(f"Log error: {e}")
            print(message)
    
    def convert_files(self):
        """Execute conversion process with progress bar"""
        csv_folder = self.csv_folder_var.get()
        
        if not csv_folder:
            messagebox.showerror("Error", "Please select CSV output folder")
            return
        
        if not self.selected_pi_files and not self.selected_engagement_files:
            messagebox.showwarning("Warning", "No files selected")
            return
        
        os.makedirs(csv_folder, exist_ok=True)
        
        self.log_text.delete(1.0, tk.END)
        self.log("Starting conversion...")
        
        # Show progress bar
        self.conversion_status_label.config(text="Converting files... Please wait")
        self.conversion_status_label.pack(pady=5)
        self.conversion_progress.pack(pady=5)
        self.conversion_progress.start(10)
        self.root.update_idletasks()
        
        try:
            converter = SmartConverter(csv_folder, log_callback=self.log)
            
            if self.selected_pi_files:
                self.conversion_status_label.config(text=f"Processing {len(self.selected_pi_files)} PI file(s)...")
                self.root.update_idletasks()
                converter.combine_pi_batches(self.selected_pi_files)
            
            if self.selected_engagement_files:
                self.conversion_status_label.config(text=f"Processing {len(self.selected_engagement_files)} engagement file(s)...")
                self.root.update_idletasks()
                converter.convert_engagement_files(self.selected_engagement_files)
            
            # Stop progress bar
            self.conversion_progress.stop()
            self.conversion_progress.pack_forget()
            self.conversion_status_label.config(text="✓ Conversion Complete!", fg='green')
            
            self.log("\n✅ Conversion Complete!")
            
            messagebox.showinfo("Success", 
                f"Conversion complete!\n\n"
                f"Processed:\n"
                f"• {len(self.selected_pi_files)} PI file(s)\n"
                f"• {len(self.selected_engagement_files)} engagement file(s)\n\n"
                f"Output: {csv_folder}")
            
            # Hide status after 3 seconds
            self.root.after(3000, lambda: self.conversion_status_label.pack_forget())
            
        except Exception as e:
            self.conversion_progress.stop()
            self.conversion_progress.pack_forget()
            self.conversion_status_label.config(text="❌ Conversion Failed", fg='red')
            
            self.log(f"\n❌ Error: {str(e)}")
            import traceback
            self.log(traceback.format_exc())
            messagebox.showerror("Error", f"Conversion failed: {str(e)}")
    
    def run_analysis(self):
        """Execute analysis with all settings and progress bar"""
        csv_folder = self.csv_folder_var.get()
        results_folder = self.results_folder_var.get()
        
        if not csv_folder or not results_folder:
            messagebox.showerror("Error", "Please select CSV and results folders")
            return
        
        if not os.path.exists(csv_folder):
            messagebox.showerror("Error", "CSV folder does not exist. Run conversion first.")
            return
        
        # Check if CSV folder has files
        csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]
        if not csv_files:
            messagebox.showerror("Error", "No CSV files found in folder. Run conversion first.")
            return
        
        os.makedirs(results_folder, exist_ok=True)
        
        self.log_text.delete(1.0, tk.END)
        
        # Show progress bar
        self.analysis_status_label.config(text="Running analysis... Please wait", fg='blue')
        self.analysis_status_label.pack(pady=5)
        self.analysis_progress.pack(pady=5)
        self.analysis_progress.start(10)
        self.root.update_idletasks()
        
        try:
            # Get all settings with validation
            try:
                n_clusters = int(self.clusters_var.get())
                if n_clusters < 3 or n_clusters > 10:
                    raise ValueError("Clusters must be between 3 and 10")
            except ValueError as e:
                self.analysis_progress.stop()
                self.analysis_progress.pack_forget()
                self.analysis_status_label.pack_forget()
                messagebox.showerror("Error", f"Invalid cluster count: {e}")
                return
            
            try:
                high_threshold = int(self.high_threshold_var.get())
                low_threshold = int(self.low_threshold_var.get())
                if high_threshold <= low_threshold:
                    raise ValueError("High threshold must be greater than low threshold")
            except ValueError as e:
                self.analysis_progress.stop()
                self.analysis_progress.pack_forget()
                self.analysis_status_label.pack_forget()
                messagebox.showerror("Error", f"Invalid threshold values: {e}")
                return
            
            eligibility_mode = str(self.eligibility_var.get())
            require_engagement = bool(self.require_engagement_var.get())
            naming_method = str(self.naming_method_var.get())
            show_metrics = bool(self.show_metrics_var.get())
            generate_viz = bool(self.generate_viz_var.get())
            zero_position = str(self.zero_position_var.get())
            
            self.log("🔧 Configuration:")
            self.log(f"   Eligibility Mode: {eligibility_mode}")
            self.log(f"   Require Engagement: {require_engagement}")
            self.log(f"   Number of Clusters: {n_clusters}")
            self.log(f"   Zero Data Position: {zero_position}")
            self.log(f"   Naming Method: {naming_method}")
            self.log(f"   Thresholds: High={high_threshold}, Low={low_threshold}")
            
            self.analysis_status_label.config(text="Initializing engine...")
            self.root.update_idletasks()
            
            # Save settings
            self.save_current_settings()
            
            # Prepare selected files list
            selected_files = list(self.selected_csv_files) if self.selected_csv_files else []
            
            self.log("\n🚀 Creating analysis engine...")
            
            # Create engine with ALL parameters
            engine = PSImsEngine(
                csv_folder=csv_folder,
                output_folder=results_folder,
                log_callback=self.log,
                eligibility_mode=eligibility_mode,
                require_engagement=require_engagement,
                naming_method=naming_method,
                high_threshold=high_threshold,
                low_threshold=low_threshold,
                show_quality_metrics=show_metrics,
                generate_visualizations=generate_viz,
                zero_data_position=zero_position,
                selected_csv_files=selected_files
            )
            
            self.log("✓ Engine initialized successfully")
            
            self.analysis_status_label.config(text="Loading data files...")
            self.root.update_idletasks()
            
            self.log("\n⏳ Starting analysis pipeline...")
            
            output_file, profiles_file, summary_file, viz_file = engine.run_analysis(n_clusters)
            
            # Stop progress bar
            self.analysis_progress.stop()
            self.analysis_progress.pack_forget()
            self.analysis_status_label.config(text="✓ Analysis Complete!", fg='green')
            
            if output_file:
                msg = (f"Analysis Complete!\n\n"
                      f"Results saved to:\n{results_folder}\n\n"
                      f"Files generated:\n"
                      f"• {os.path.basename(output_file)}\n")
                
                if profiles_file:
                    msg += f"• {os.path.basename(profiles_file)}\n"
                if summary_file:
                    msg += f"• {os.path.basename(summary_file)}\n"
                if viz_file:
                    msg += f"• {os.path.basename(viz_file)}\n"
                
                messagebox.showinfo("Success", msg)
            else:
                messagebox.showwarning("Warning", "Analysis completed but no output generated")
            
            # Hide status after 3 seconds
            self.root.after(3000, lambda: self.analysis_status_label.pack_forget())
                
        except Exception as e:
            self.analysis_progress.stop()
            self.analysis_progress.pack_forget()
            self.analysis_status_label.config(text="❌ Analysis Failed", fg='red')
            
            error_msg = str(e)
            error_type = type(e).__name__
            
            # Print to console for Jupyter
            print(f"\n{'='*60}")
            print(f"❌ ERROR OCCURRED")
            print(f"Type: {error_type}")
            print(f"Message: {error_msg}")
            print(f"{'='*60}\n")
            
            import traceback
            error_trace = traceback.format_exc()
            print("Full Traceback:")
            print(error_trace)
            print(f"{'='*60}\n")
            
            # Also log to GUI
            self.log(f"\n❌ ERROR OCCURRED")
            self.log(f"   Type: {error_type}")
            self.log(f"   Message: {error_msg}")
            self.log("\n📋 Full Traceback:")
            self.log(error_trace)
            
            messagebox.showerror("Analysis Failed", 
                               f"Error Type: {error_type}\n\n"
                               f"Error: {error_msg}\n\n"
                               f"See Jupyter console for full traceback.")


# =====================================================
# MAIN EXECUTION
# =====================================================

def main():
    """Launch PSIMS application"""
    root = tk.Tk()
    app = PSImsGUI(root)
    root.mainloop()

if __name__ == "__main__":
    main()

AttributeError: 'PSImsGUI' object has no attribute 'browse_input'